In [1]:
import torch
import pickle
import numpy as np
import pandas as pd
import os
import json 
from os.path import dirname



root_path = dirname(os.getcwd()) + "/SEPH_TIME"

pd.set_option("display.max_columns", None)
data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/processed/"
data_dir_graphs = root_path + "/data/datasets/graphs_repair/"

print(root_path, data_dir, data_dir_processed, data_dir_graphs, sep="\n")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# device = "cpu"

/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/original/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/processed/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_TIME/data/datasets/graphs_repair/


In [2]:
with open("data/dataset_features.json", 'r') as file:
    datasets_info = json.load(file)

In [3]:
list(datasets_info.keys())

['BPI12_DECLINED_COMPLETE',
 'sepsis_cases_1',
 'sepsis_cases_4',
 'BPIC15_common',
 'BPIC15_4_f2',
 'bpic2012_O_ACCEPTED-COMPLETE']

In [4]:
dataset = "bpic2012_O_ACCEPTED-COMPLETE"

In [5]:
if dataset == "BPIC15_4_f2":
    with open("data/dataset_features.json", 'r') as file:
        dataset_info = json.load(file)[dataset]
elif dataset.startswith("BPIC15"):
    with open("data/dataset_features.json", 'r') as file:
        dataset_info = json.load(file)["BPIC15_common"]
else:
    with open("data/dataset_features.json", 'r') as file:
        dataset_info = json.load(file)[dataset]

In [6]:
categorical_columns = dataset_info["categorical"]
real_value_columns = dataset_info["numerical"]

In [7]:
tab_all = pd.read_csv(data_dir_processed+dataset+"_processed_all.csv")
tab_all.head()

/tmp/ipykernel_4763/3535436184.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  tab_all = pd.read_csv(data_dir_processed+dataset+"_processed_all.csv")


,AMOUNT_REQ,CaseID,label,Activity,Resource,lifecycle:transition,timesincemidnight,timesincelastevent,timesincecasestart,event_nr,month,weekday,hour,open_cases,time:timestamp,remaining_time
0,20000,173688,deviant,A_SUBMITTED-COMPLETE,112.0,COMPLETE,98,0.000000,0.000000,1,10,5,1,1,1.317419e+09,1072732.480
1,20000,173688,deviant,A_PARTLYSUBMITTED-COMPLETE,112.0,COMPLETE,98,0.005567,0.005567,2,10,5,1,1,1.317419e+09,1072732.146
2,20000,173688,deviant,A_PREACCEPTED-COMPLETE,112.0,COMPLETE,99,0.883767,0.889333,3,10,5,1,1,1.317419e+09,1072679.120
3,20000,173688,deviant,W_Completeren aanvraag-SCHEDULE,112.0,SCHEDULE,99,0.016150,0.905483,4,10,5,1,1,1.317419e+09,1072678.151
4,20000,173688,deviant,W_Completeren aanvraag-START,112.0,START,756,657.126033,658.031517,5,10,5,12,10,1.317458e+09,1033250.589


In [8]:
import pandas as pd

tab_test = pd.read_csv(data_dir_processed + f"{dataset}_processed_test.csv")

# time:timestamp should be in float representing number of seconds.
durations = (
    tab_test
    .groupby("CaseID")["time:timestamp"]
    .agg(start="min", end="max")
)
durations["duration_sec"] = (durations["end"] - durations["start"])

# 4) Compute averages
avg_sec  = durations["duration_sec"].mean()
avg_days = avg_sec / (24 * 3600)

print(f"Average trace duration (test set): {avg_sec:,.0f} seconds\n")
print(f"which means ≃ {avg_days:.1f} days")


Average trace duration (test set): 1,325,519 seconds

which means ≃ 15.3 days


In [9]:
import random

torch.manual_seed(0)
torch.cuda.manual_seed(0)
random.seed(0)
np.random.seed(0)

In [10]:
with open(data_dir_graphs + dataset + "_TRAIN_repair.pkl", "rb") as f:
    X_train = pickle.load(f)
with open(data_dir_graphs + dataset + "_VALID_repair.pkl", "rb") as f:
    X_valid = pickle.load(f)
#with open(data_dir_graphs + dataset + "_TEST4_repair.pkl", "rb") as f:
#    X_test = pickle.load(f)

MaxPrefix = 20

X_tests = {}
for L in range(1, MaxPrefix + 1):
    fname = f"{dataset}_TEST{L}_repair.pkl"
    path  = os.path.join(data_dir_graphs, fname)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Expected file not found: {path}")
    with open(path, "rb") as f:
        X_tests[L] = pickle.load(f)
    print(f"Loaded {len(X_tests[L])} graphs for prefix length {L} from {fname}")

Loaded 937 graphs for prefix length 1 from bpic2012_O_ACCEPTED-COMPLETE_TEST1_repair.pkl
Loaded 937 graphs for prefix length 2 from bpic2012_O_ACCEPTED-COMPLETE_TEST2_repair.pkl
Loaded 937 graphs for prefix length 3 from bpic2012_O_ACCEPTED-COMPLETE_TEST3_repair.pkl
Loaded 937 graphs for prefix length 4 from bpic2012_O_ACCEPTED-COMPLETE_TEST4_repair.pkl
Loaded 937 graphs for prefix length 5 from bpic2012_O_ACCEPTED-COMPLETE_TEST5_repair.pkl
Loaded 937 graphs for prefix length 6 from bpic2012_O_ACCEPTED-COMPLETE_TEST6_repair.pkl
Loaded 937 graphs for prefix length 7 from bpic2012_O_ACCEPTED-COMPLETE_TEST7_repair.pkl
Loaded 937 graphs for prefix length 8 from bpic2012_O_ACCEPTED-COMPLETE_TEST8_repair.pkl
Loaded 937 graphs for prefix length 9 from bpic2012_O_ACCEPTED-COMPLETE_TEST9_repair.pkl
Loaded 937 graphs for prefix length 10 from bpic2012_O_ACCEPTED-COMPLETE_TEST10_repair.pkl
Loaded 937 graphs for prefix length 11 from bpic2012_O_ACCEPTED-COMPLETE_TEST11_repair.pkl
Loaded 937 graphs

In [11]:

from torch_geometric.data import Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import ToUndirected, NormalizeFeatures, Compose

#transform = ToUndirected()
transform = Compose([ToUndirected(), NormalizeFeatures()])

with torch.no_grad():
        for i in range(len(X_train)):
                X_train[i] = transform(X_train[i])
        for i in range(len(X_valid)):
                X_valid[i] = transform(X_valid[i])
        for L, graphs_L in X_tests.items():
                for i in range(len(graphs_L)):
                        graphs_L[i] = transform(graphs_L[i])
    


In [12]:
edge_types = set()
node_types = set()
for i in range(len(X_train)):
    n, edge_type = X_train[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
for i in range(len(X_valid)):
    n, edge_type = X_valid[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
for graphs_L in X_tests.values():
    for i in range(len(graphs_L)):
        n, edge_type = graphs_L[i].metadata()
        for x in n:
            node_types.add(x)
        for x in edge_type:
            edge_types.add(x)




In [13]:
node_types = list(node_types)
edge_types = list(edge_types)

In [14]:
node_types

['lifecycle:transition',
 'event_nr',
 'timesincelastevent',
 'timesincemidnight',
 'timesincecasestart',
 'month',
 'hour',
 'label',
 'Resource',
 'time:timestamp',
 'open_cases',
 'AMOUNT_REQ',
 'Activity',
 'weekday']

In [15]:
edge_types

[('Activity', 'related_to', 'timesincecasestart'),
 ('lifecycle:transition', 'rev_related_to', 'Activity'),
 ('timesincecasestart', 'rev_related_to', 'Activity'),
 ('month', 'related_to', 'month'),
 ('timesincelastevent', 'related_to', 'timesincelastevent'),
 ('weekday', 'rev_related_to', 'Activity'),
 ('Activity', 'related_to', 'event_nr'),
 ('timesincemidnight', 'rev_related_to', 'Activity'),
 ('Resource', 'related_to', 'Resource'),
 ('Activity', 'related_to', 'Resource'),
 ('Activity', 'related_to', 'weekday'),
 ('Activity', 'related_to', 'AMOUNT_REQ'),
 ('Activity', 'related_to', 'time:timestamp'),
 ('hour', 'rev_related_to', 'Activity'),
 ('Activity', 'related_to', 'label'),
 ('Activity', 'related_to', 'lifecycle:transition'),
 ('label', 'related_to', 'label'),
 ('AMOUNT_REQ', 'related_to', 'AMOUNT_REQ'),
 ('event_nr', 'related_to', 'event_nr'),
 ('timesincecasestart', 'related_to', 'timesincecasestart'),
 ('label', 'rev_related_to', 'Activity'),
 ('open_cases', 'rev_related_to', 

## Hyperopt

In [16]:
from ax.service.managed_loop import optimize

In [17]:
from torch_geometric.nn import (
    HeteroConv,
    global_mean_pool,
    GATv2Conv,
    SAGEConv,
    TransformerConv
)
from torch.nn import (
    ModuleList,
    Module,
    Linear
  )
from typing_extensions import Self

In [18]:
from torch.nn import Module, ModuleList, Linear
import torch.nn as nn
from torch_geometric.nn import HeteroConv, SAGEConv, global_mean_pool

class HGNN(Module):
    def __init__(self, node_types, edge_types, parameters):
        super().__init__()
        hid = parameters["hid"]
        layers = parameters["layers"]
        aggregation = parameters["aggregation"]
        
        self.node_types = node_types
        self.edge_types = edge_types
        
        # Convolutional layers for heterogeneous graph
        self.convs = ModuleList()
        for _ in range(layers):
            conv = HeteroConv(
                {relation: SAGEConv((-1, -1), hid, aggr=aggregation)
                 for relation in edge_types},
                aggr=aggregation
            )
            self.convs.append(conv)
        
        # Linear layer for graph-level prediction
        self.lin = Linear(len(node_types) * hid, 1)
    
    def forward(self, batch):
        x_dict = batch.x_dict
        edge_index_dict = batch.edge_index_dict
        
        # Apply convolutional layers
        for conv in self.convs:
            x_dict = conv(x_dict, edge_index_dict)
            x_dict = {key: x.relu() for key, x in x_dict.items()}
        
        # Pool node features for each node type
        graph_features = []
        for node_type in self.node_types:
            x = x_dict[node_type]
            batch_idx = batch[node_type].batch  # Batch index for pooling
            pooled = global_mean_pool(x, batch_idx)
            graph_features.append(pooled)
        
        # Concatenate pooled features
        graph_features = torch.cat(graph_features, dim=-1)
        
        # Predict remaining time
        output = self.lin(graph_features).squeeze(-1)
        return output  # Shape: [batch_size]

In [19]:
from torcheval.metrics.functional import multiclass_accuracy, multiclass_f1_score

In [20]:
import torch.nn as nn

In [21]:
import time

In [22]:
from torch_geometric.data import DataLoader
from copy import deepcopy

def train_hgnn(config, node_types, edge_types, epochs=50):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net = HGNN(node_types=node_types, edge_types=edge_types, parameters=config).to(device)
    loss_fn = nn.L1Loss()  # MAE for regression
    optimizer = torch.optim.Adam(net.parameters(), lr=config["lr"])
    
    train_loader = DataLoader(X_train, batch_size=config["batch_size"], shuffle=True)
    valid_loader = DataLoader(X_valid, batch_size=config["batch_size"], shuffle=False)
    
    best_model = None
    best_loss = float("inf")
    patience = 5
    pat_count = 0
    
    for epoch in range(epochs):
        net.train()
        train_losses = []
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            preds = net(batch)  # Graph-level predictions
            true = batch.y  # Graph-level targets
            loss = loss_fn(preds, true)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
        
        avg_train_loss = sum(train_losses) / len(train_losses)
        
        net.eval()
        valid_losses = []
        with torch.no_grad():
            for batch in valid_loader:
                batch = batch.to(device)
                preds = net(batch)
                true = batch.y
                valid_losses.append(loss_fn(preds, true).item())
        
        avg_val_loss = sum(valid_losses) / len(valid_losses)
        
        print(f"Epoch {epoch+1}/{epochs}, Train MAE: {avg_train_loss:.4f}, Valid MAE: {avg_val_loss:.4f}")
        
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            best_model = deepcopy(net)
            pat_count = 0
        else:
            pat_count += 1
            if pat_count >= patience:
                print("Early stopping")
                break
    
    return best_model

In [23]:
'''
def test_hgnn(net):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    test_loader = DataLoader(X_test, batch_size=128, shuffle=False)
    all_preds = []
    all_targets = []
    net.eval()
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            preds = net(batch)
            all_preds.append(preds)
            all_targets.append(batch.y)
    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    mae = nn.L1Loss()(all_preds, all_targets).item()
    print(f"Test MAE: {mae:.4f}")
    return {"remaining_time_mae": mae}
'''

def test_hgnn(net):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net.eval()

    from sklearn.metrics import mean_absolute_error
    import numpy as np

    prefix_maes = []
    prefix_counts = []

    for L in range(1, MaxPrefix + 1):
        graphs = X_tests[L]
        if len(graphs) == 0:
            continue

        loader = DataLoader(graphs, batch_size=128, shuffle=False)
        y_true = []
        y_pred = []

        with torch.no_grad():
            for batch in loader:
                batch = batch.to(device)
                preds = net(batch)
                y_pred.extend(preds.cpu().numpy())
                y_true.extend(batch.y.cpu().numpy())

        mae = mean_absolute_error(y_true, y_pred)
        prefix_maes.append(mae)
        prefix_counts.append(len(y_true))

    maes = np.array(prefix_maes)
    counts = np.array(prefix_counts)
    total = counts.sum()

    weighted_mae = np.average(maes, weights=counts)

    weighted_var = np.average((maes - weighted_mae) ** 2, weights=counts)
    weighted_std = np.sqrt(weighted_var)

    weighted_mae /= 86400
    weighted_std /= 86400

    print(f"Weighted MAE (days): {weighted_mae:.4f} ± {weighted_std:.4f}")
    
    return {
        "remaining_time_mae": weighted_mae, #conversion to days.
        "remaining_time_std": weighted_std,
    }


In [24]:
outputreal = ["remaining_time"]
print(outputreal)

['remaining_time']


In [25]:
def train_evaluate(config):
    trained_net = train_hgnn(config, node_types=node_types,edge_types=edge_types, epochs=50)
    return test_hgnn(trained_net)

In [26]:
import logging

logging.getLogger("root").setLevel(logging.ERROR)

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [27]:
sample_config = {
    "hid":          128, #128
    "layers":       2, #2
    "lr":           1e-3,
    "batch_size":   256, #256
    "aggregation": "mean",
}

# 1-epoch train just to exercise the code-path and see prints
net = train_hgnn(sample_config,node_types=node_types,edge_types=edge_types, epochs=1)

# run the test (with your debug prints enabled)
res = test_hgnn(net)
print("test_hgnn returned:", res)

Epoch 1/1, Train MAE: 1682260.7396, Valid MAE: 1666046.3750
Weighted MAE (days): 13.6675 ± 1.9830
test_hgnn returned: {'remaining_time_mae': 13.667483596206763, 'remaining_time_std': 1.9830437724944547}


In [28]:
from ax import Metric
tracking_metrics = [Metric(name="remaining_time_std")]

best_parameters, values, experiment, model = optimize(
    parameters=[
        {"name": "hid", "type": "choice", "values": [128], "value_type": "int", "is_ordered" : True,"sort_values":False},
        #{"name": "layers", "type": "choice", "values": [2, 3, 4, 5], "value_type": "int", "is_ordered" : True, "sort_values":False},
        {"name": "layers", "type": "choice", "values": [2], "value_type": "int", "is_ordered" : True, "sort_values":False},
        {"name": "lr", "type": "range", "bounds": [1e-4, 1e-1], "value_type": "float", "log_scale": True},
        {"name": "batch_size", "type": "choice", "values": [128,256,512], "value_type": "int", "is_ordered" : True,"sort_values":False}, 
        
        #{"name": "heads", "type": "choice", "values": [1,2], "value_type": "int", "is_ordered" : True,"sort_values":False},
        #{"name": "heads", "type": "choice", "values": [1], "value_type": "int", "is_ordered" : True,"sort_values":False},
        
        {"name": "aggregation", "type" : "choice", "values" :["sum", "mean", "max"], "value_type" : "str"}
        #{"name": "aggregation", "type" : "choice", "values" :["max"], "value_type" : "str"},
     
    ],
  
    evaluation_function=lambda config:train_evaluate({**config,"nodes_relations": edge_types}),
    objective_name='remaining_time_mae', 
    arms_per_trial=1,
    minimize = True,
    random_seed = 123,
    total_trials = 15
)

print("Best parameters:", best_parameters)
means, covariances = values
print("Means", means)
print("Experiment:", experiment)

[INFO 07-22 13:45:37] ax.service.utils.instantiation: Choice parameter hid contains only one value, converting to a fixed parameter instead.
[INFO 07-22 13:45:37] ax.service.utils.instantiation: Choice parameter layers contains only one value, converting to a fixed parameter instead.
/home/matteo/Documents/GNN-test2/SEPH_MODELS/env3/lib/python3.10/site-packages/ax/service/utils/instantiation.py:248: AxParameterWarning: `is_ordered` is not specified for `ChoiceParameter` "aggregation". Defaulting to `False`  since the parameter is a string with more than 2 choices.. To override this behavior (or avoid this warning), specify `is_ordered` during `ChoiceParameter` construction. Note that choice parameters with exactly 2 choices are always considered ordered and that the user-supplied `is_ordered` has no effect in this particular case.
  return ChoiceParameter(
/home/matteo/Documents/GNN-test2/SEPH_MODELS/env3/lib/python3.10/site-packages/ax/service/utils/instantiation.py:248: AxParameterWa

Epoch 1/50, Train MAE: 1681963.9792, Valid MAE: 1666052.8750
Epoch 2/50, Train MAE: 1679221.6042, Valid MAE: 1666051.6250
Epoch 3/50, Train MAE: 1683481.2500, Valid MAE: 1666050.2083
Epoch 4/50, Train MAE: 1679432.6458, Valid MAE: 1666048.1250
Epoch 5/50, Train MAE: 1681562.0625, Valid MAE: 1666045.4167
Epoch 6/50, Train MAE: 1680700.6250, Valid MAE: 1666041.8333
Epoch 7/50, Train MAE: 1682794.8333, Valid MAE: 1666037.2917
Epoch 8/50, Train MAE: 1682007.5312, Valid MAE: 1666031.5833
Epoch 9/50, Train MAE: 1681901.0417, Valid MAE: 1666024.6250
Epoch 10/50, Train MAE: 1679490.8958, Valid MAE: 1666016.2917
Epoch 11/50, Train MAE: 1682055.6979, Valid MAE: 1666006.2083
Epoch 12/50, Train MAE: 1682630.4896, Valid MAE: 1665994.2083
Epoch 13/50, Train MAE: 1682230.5000, Valid MAE: 1665980.2500
Epoch 14/50, Train MAE: 1681366.5000, Valid MAE: 1665964.1667
Epoch 15/50, Train MAE: 1683166.8854, Valid MAE: 1665945.5417
Epoch 16/50, Train MAE: 1679788.9271, Valid MAE: 1665924.4167
Epoch 17/50, Trai

[INFO 07-22 13:54:48] ax.core.experiment: Attached data has some metrics ({'remaining_time_std'}) that are not among the metrics on this experiment. Note that attaching data will not automatically add those metrics to the experiment. For these metrics to be automatically fetched by `experiment.fetch_data`, add them via `experiment.add_tracking_metric` or update the experiment's optimization config.
[INFO 07-22 13:54:48] ax.service.managed_loop: Running optimization trial 2...
[ERROR 07-22 13:54:48] ax.core.observation: Data contains metric remaining_time_std that has not been added to the experiment. You can either update the `optimization_config` or attach it as a tracking metric using `Experiment.add_tracking_metrics` or `AxClient.add_tracking_metrics`. Ignoring all data for metric remaining_time_std.
NoneType: None


Weighted MAE (days): 13.6276 ± 1.9810
Epoch 1/50, Train MAE: 1260494.7760, Valid MAE: 741442.6667
Epoch 2/50, Train MAE: 791985.9219, Valid MAE: 714244.3229
Epoch 3/50, Train MAE: 776787.6432, Valid MAE: 711715.2188
Epoch 4/50, Train MAE: 784343.8854, Valid MAE: 716961.0104
Epoch 5/50, Train MAE: 785304.0938, Valid MAE: 710511.1250
Epoch 6/50, Train MAE: 784435.9141, Valid MAE: 709078.3750
Epoch 7/50, Train MAE: 780015.0104, Valid MAE: 707119.4271
Epoch 8/50, Train MAE: 776020.1823, Valid MAE: 711226.6562
Epoch 9/50, Train MAE: 773435.8177, Valid MAE: 714607.5938
Epoch 10/50, Train MAE: 776531.4974, Valid MAE: 708058.0729
Epoch 11/50, Train MAE: 775846.4635, Valid MAE: 737287.8958
Epoch 12/50, Train MAE: 778432.1120, Valid MAE: 707855.5417
Early stopping


[INFO 07-22 13:57:56] ax.core.experiment: Attached data has some metrics ({'remaining_time_std'}) that are not among the metrics on this experiment. Note that attaching data will not automatically add those metrics to the experiment. For these metrics to be automatically fetched by `experiment.fetch_data`, add them via `experiment.add_tracking_metric` or update the experiment's optimization config.
[INFO 07-22 13:57:56] ax.service.managed_loop: Running optimization trial 3...
[ERROR 07-22 13:57:56] ax.core.observation: Data contains metric remaining_time_std that has not been added to the experiment. You can either update the `optimization_config` or attach it as a tracking metric using `Experiment.add_tracking_metrics` or `AxClient.add_tracking_metrics`. Ignoring all data for metric remaining_time_std.
NoneType: None
[ERROR 07-22 13:57:56] ax.core.observation: Data contains metric remaining_time_std that has not been added to the experiment. You can either update the `optimization_con

...

Weighted MAE (days): 7.1825 ± 1.0218
Epoch 1/50, Train MAE: 1681826.8958, Valid MAE: 1649577.0625
Epoch 2/50, Train MAE: 1682007.4792, Valid MAE: 1648765.1875
Epoch 3/50, Train MAE: 1680484.5833, Valid MAE: 1646806.7500
Epoch 4/50, Train MAE: 1675646.3750, Valid MAE: 1642768.1875
Epoch 5/50, Train MAE: 1671630.2708, Valid MAE: 1635292.5000
Epoch 6/50, Train MAE: 1661820.1458, Valid MAE: 1622533.8750
Epoch 7/50, Train MAE: 1648956.9792, Valid MAE: 1602213.4375
Epoch 8/50, Train MAE: 1620715.5417, Valid MAE: 1571668.0000
Epoch 9/50, Train MAE: 1587229.8542, Valid MAE: 1527863.0625
Epoch 10/50, Train MAE: 1537023.1875, Valid MAE: 1467359.9375
Epoch 11/50, Train MAE: 1471170.7708, Valid MAE: 1386688.6250
Epoch 12/50, Train MAE: 1379990.0417, Valid MAE: 1283220.3750
Epoch 13/50, Train MAE: 1268747.1042, Valid MAE: 1153070.6250
Epoch 14/50, Train MAE: 1133383.7812, Valid MAE: 1001368.2188
Epoch 15/50, Train MAE: 990432.8021, Valid MAE: 860969.3750
Epoch 16/50, Train MAE: 876465.1250, Valid M

[INFO 07-22 14:57:13] ax.core.experiment: Attached data has some metrics ({'remaining_time_std'}) that are not among the metrics on this experiment. Note that attaching data will not automatically add those metrics to the experiment. For these metrics to be automatically fetched by `experiment.fetch_data`, add them via `experiment.add_tracking_metric` or update the experiment's optimization config.
[ERROR 07-22 14:57:13] ax.core.observation: Data contains metric remaining_time_std that has not been added to the experiment. You can either update the `optimization_config` or attach it as a tracking metric using `Experiment.add_tracking_metrics` or `AxClient.add_tracking_metrics`. Ignoring all data for metric remaining_time_std.
NoneType: None
[ERROR 07-22 14:57:13] ax.core.observation: Data contains metric remaining_time_std that has not been added to the experiment. You can either update the `optimization_config` or attach it as a tracking metric using `Experiment.add_tracking_metrics` 

Weighted MAE (days): 6.6572 ± 0.7506


[WARNING 07-22 14:57:19] ax.modelbridge.cross_validation: Metric remaining_time_mae was unable to be reliably fit.
[WARNING 07-22 14:57:19] ax.service.utils.best_point: Model fit is poor; falling back on raw data for best point.
[WARNING 07-22 14:57:19] ax.service.utils.best_point: Model fit is poor and data on objective metric remaining_time_mae is noisy; interpret best points results carefully.


Best parameters: {'lr': 0.004929750755324542, 'batch_size': 512, 'aggregation': 'sum', 'hid': 128, 'layers': 2}
Means {'remaining_time_mae': 6.654755412848938, 'remaining_time_std': 0.7445266221587916}
Experiment: Experiment(None)


In [29]:
from ax.service.utils.report_utils import exp_to_df
results = exp_to_df(experiment)

[WARNING 07-22 14:57:19] ax.service.utils.report_utils: Column reason missing for all trials. Not appending column.


In [30]:
results = results.sort_values(by="remaining_time_mae")
results.to_csv(f"results/{dataset}.csv", sep=",")